In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split



url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(url)



print(df.head())

print(df.shape)

print(df.columns.tolist())


df.columns = df.columns.str.lower().str.replace(' ', '_')

print(df.columns.tolist())




print(df.isnull().sum())



df['totalcharges'] = pd.to_numeric(
    df['totalcharges'],
    errors='coerce'
)


print("Missing totalcharges:",
      df['totalcharges'].isnull().sum())



print(
    df[df['totalcharges'].isnull()]
    [['customerid', 'totalcharges']]
)



df['totalcharges'] = df['totalcharges'].fillna(0)


# Verify
print("Missing totalcharges after filling:",
      df['totalcharges'].isnull().sum())




print(df['churn'].value_counts())




df['churn'] = (df['churn'] == 'Yes').astype(int)



print(df['churn'].value_counts())



df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)


print(
    len(df_full_train),
    len(df_test)
)



df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)


print(
    len(df_train),
    len(df_val),
    len(df_test)
)


df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values




del df_train['churn']
del df_val['churn']
del df_test['churn']



print("Training rows:", len(df_train))
print("Validation rows:", len(df_val))
print("Test rows:", len(df_test))

print("y_train:", len(y_train))
print("y_val:", len(y_val))
print("y_test:", len(y_test))

print("\nTarget distribution:")
print(pd.Series(y_train).value_counts())

df_full_train = df_full_train.reset_index(drop=True)

print("\nMissing values:")
print(df_full_train.isnull().sum())

print("\nChurn counts:")
print(df_full_train.churn.value_counts())

print("\nChurn distribution:")
print(df_full_train.churn.value_counts(normalize=True))

# Global churn rate
global_churn = df_full_train.churn.mean()

print("\nGlobal churn rate:")
print(global_churn)

# Numerical variables
numerical = [
    'tenure',
    'monthlycharges',
    'totalcharges'
]

# Categorical variables
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod'
]

print("\nNumerical variables:")
print(numerical)

print("\nCategorical variables:")
print(categorical)

print("\nNumber of unique values:")
print(df_full_train[categorical].nunique())


# ============================================================
# 3.5 - FEATURE IMPORTANCE
#       CHURN RATE AND RISK RATIO
# ============================================================

global_churn = df_full_train.churn.mean()

print("\nGlobal churn:")
print(global_churn)

# Example: gender
df_group = (
    df_full_train
    .groupby('gender')
    .churn
    .agg(['mean', 'count'])
)

df_group.columns = [
    'churn_rate',
    'count'
]

df_group['risk'] = (
    df_group.churn_rate / global_churn
)

print("\nGender churn:")
print(df_group)


# Function for calculating churn rate and risk
def calculate_risk(df, column):

    result = (
        df
        .groupby(column)
        .churn
        .agg(['mean', 'count'])
    )

    result.columns = [
        'churn_rate',
        'count'
    ]

    result['risk'] = (
        result.churn_rate / df.churn.mean()
    )

    return result


print("\nContract:")
print(
    calculate_risk(
        df_full_train,
        'contract'
    )
)

print("\nInternet Service:")
print(
    calculate_risk(
        df_full_train,
        'internetservice'
    )
)

print("\nPayment Method:")
print(
    calculate_risk(
        df_full_train,
        'paymentmethod'
    )
)

print("\nGender:")
print(
    calculate_risk(
        df_full_train,
        'gender'
    )
)


# ============================================================
# 3.6 - FEATURE IMPORTANCE
#       MUTUAL INFORMATION
# ============================================================

from sklearn.metrics import mutual_info_score


def mutual_info_churn_score(series):

    return mutual_info_score(
        series,
        df_full_train.churn
    )


mi = df_full_train[
    categorical
].apply(
    mutual_info_churn_score
)

print("\nMutual Information:")
print(
    mi.sort_values(
        ascending=False
    )
)

print("\nMutual Information DataFrame:")
print(
    mi
    .sort_values(ascending=False)
    .to_frame(name='MI')
)


# ============================================================
# 3.7 - FEATURE IMPORTANCE
#       CORRELATION
# ============================================================

print("\nCorrelation with Churn:")
print(
    df_full_train[
        numerical
    ].corrwith(
        df_full_train.churn
    )
)

print("\nSorted correlation:")
print(
    df_full_train[
        numerical
    ]
    .corrwith(
        df_full_train.churn
    )
    .sort_values()
)

print("\nFull correlation matrix:")
print(
    df_full_train[
        numerical + ['churn']
    ].corr()
)


# ============================================================
# 3.8 - ONE-HOT ENCODING
# ============================================================

from sklearn.feature_extraction import DictVectorizer

# Convert training data to dictionaries
train_dict = (
    df_train[
        categorical + numerical
    ]
    .to_dict(
        orient='records'
    )
)

# Convert validation data to dictionaries
val_dict = (
    df_val[
        categorical + numerical
    ]
    .to_dict(
        orient='records'
    )
)

# Create DictVectorizer
dv = DictVectorizer(
    sparse=False
)

# Fit and transform training data
X_train = dv.fit_transform(
    train_dict
)

# Transform validation data
X_val = dv.transform(
    val_dict
)

print("\nX_train shape:")
print(X_train.shape)

print("\nX_val shape:")
print(X_val.shape)

print("\nFeature names:")
print(
    dv.get_feature_names_out()
)

print("\nFirst 5 rows of X_train:")
print(X_train[:5])




def sigmoid(z):

    return 1 / (
        1 + np.exp(-z)
    )



z = np.array([
    -2,
    -1,
    0,
    1,
    2
])

print("\nSigmoid values:")
print(sigmoid(z))


score = 0.5

print("\nSigmoid(0.5):")
print(sigmoid(score))




from sklearn.linear_model import LogisticRegression

# Create model
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000
)


model.fit(
    X_train,
    y_train
)

y_pred = model.predict_proba(
    X_val
)[:, 1]

print("\nFirst 10 predicted probabilities:")
print(y_pred[:10])

churn_decision = (
    y_pred >= 0.5
)

print("\nFirst 10 decisions:")
print(churn_decision[:10])


accuracy = (
    churn_decision == y_val
).mean()

print("\nValidation accuracy:")
print(accuracy)



feature_names = (
    dv.get_feature_names_out()
)

coef = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0]
})

print("\nAll coefficients:")
print(
    coef.sort_values(
        'coefficient',
        ascending=False
    )
)

print("\nTop positive coefficients:")
print(
    coef
    .sort_values(
        'coefficient',
        ascending=False
    )
    .head(10)
)

print("\nTop negative coefficients:")
print(
    coef
    .sort_values(
        'coefficient',
        ascending=True
    )
    .head(10)
)

# Intercept
print("\nIntercept:")
print(
    model.intercept_[0]
)

# Odds ratios
coef['odds_ratio'] = np.exp(
    coef['coefficient']
)

print("\nOdds ratios:")
print(
    coef.sort_values(
        'odds_ratio',
        ascending=False
    ).head(10)
)


full_train_dict = (
    df_full_train[
        categorical + numerical
    ]
    .to_dict(
        orient='records'
    )
)

test_dict = (
    df_test[
        categorical + numerical
    ]
    .to_dict(
        orient='records'
    )
)


dv = DictVectorizer(
    sparse=False
)


X_full_train = dv.fit_transform(
    full_train_dict
)


X_test = dv.transform(
    test_dict
)

print("\nFull training matrix:")
print(X_full_train.shape)

print("\nTest matrix:")
print(X_test.shape)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000
)


model.fit(
    X_full_train,
    df_full_train.churn.values
)

y_pred = model.predict_proba(
    X_test
)[:, 1]

print("\nFirst 10 test predictions:")
print(y_pred[:10])



customer = df_test.iloc[0].to_dict()

print("\nCustomer:")
print(customer)

X_customer = dv.transform(
    [customer]
)


probability = model.predict_proba(
    X_customer
)[0, 1]

print("\nChurn probability:")
print(
    f"{probability:.3f}"
)


prediction = probability >= 0.5

print("\nChurn prediction:")
print(prediction)



test_decision = (
    y_pred >= 0.5
)

test_accuracy = (
    test_decision == y_test
).mean()

print("\nFinal Test Accuracy:")
print(test_accuracy)



print("\n====================================")
print("ML ZOOMCAMP MODULE 3 COMPLETED")
print("Lessons 3.1 -> 3.12")
print("====================================")

print("Dataset shape:", df.shape)
print("Train size:", len(df_train))
print("Validation size:", len(df_val))
print("Test size:", len(df_test))

print("Validation accuracy:", accuracy)
print("Test accuracy:", test_accuracy)

print("Example customer churn probability:",
      round(probability, 3))

print("Example customer prediction:",
      "CHURN" if prediction else "NO CHURN")

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape